In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import QuantileTransformer, RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sentence_transformers import SentenceTransformer
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    'n_folds': 7,
    'random_state': 42,
    'embedding_model': 'all-MiniLM-L6-v2',
    'embedding_dims': 100,
    'tfidf_features': 150,
    'n_estimators': 5000,
    'early_stopping': 200,
}

def calculate_smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

def extract_enhanced_fields(catalog):
    if pd.isna(catalog):
        return {}, None, None

    catalog = str(catalog)
    fields = {}

    for pattern in [r'Item\s*Name\s*:\s*(.+?)(?=\n|Bullet|Product|Value|Brand|$)',
                    r'Item\s*:\s*(.+?)(?=\n|Bullet|Product|Value|Brand|$)']:
        m = re.search(pattern, catalog, re.I)
        if m:
            fields['item'] = m.group(1).strip()[:500]
            break

    bullets = re.findall(r'Bullet\s*Point\s*\d*\s*:\s*(.+?)(?=\n(?:Bullet|Product|Value|Brand|$))',
                        catalog, re.I | re.DOTALL)
    fields['bullets'] = " ".join([b.strip() for b in bullets])[:1000]

    m = re.search(r'Product\s*Description\s*:\s*(.+?)(?=\nBullet|\nValue|$)', catalog, re.I | re.DOTALL)
    fields['description'] = m.group(1).strip()[:800] if m else ""

    m = re.search(r'(?:Brand|Manufacturer)\s*:\s*([^\n]+)', catalog, re.I)
    fields['brand'] = m.group(1).strip() if m else ""

    value, unit = None, None

    m_val = re.search(r'Value\s*:\s*([\d.,]+)', catalog, re.I)
    m_unit = re.search(r'Unit\s*:\s*([^\s,;.\n]+)', catalog, re.I)

    if m_val:
        try:
            val_str = m_val.group(1).replace(',', '')
            value = float(val_str)
        except:
            pass

    if m_unit:
        unit = m_unit.group(1).strip().lower()

    if value is None or unit is None:
        search_text = fields.get('item', '') + ' ' + fields.get('bullets', '')

        patterns = [
            r'(\d+\.?\d*)\s*(oz|ounce|lb|pound|g|gram|kg|ml|liter|l|fl\s*oz|gallon|gal|qt|count|ct|pack|piece)',
            r'(\d+\.?\d*)\s*-?\s*(oz|lb|g|kg|ml|l)',
            r'(\d+)\s*pack',
        ]

        for p in patterns:
            m = re.search(p, search_text, re.I)
            if m:
                try:
                    if value is None:
                        value = float(m.group(1).replace(',', ''))
                    if unit is None and len(m.groups()) > 1:
                        unit = m.group(2).lower().strip()
                    if value is not None and unit is not None:
                        break
                except:
                    pass

    return fields, value, unit

def advanced_unit_standardization(value, unit, text_content=""):
    if pd.isna(value) or value is None:
        return None, None, 'unknown'

    try:
        value = float(value)
    except:
        return None, None, 'unknown'

    if pd.isna(unit) or unit is None:
        unit = 'unknown'

    unit = str(unit).lower().strip()
    text = text_content.lower()

    category = 'general'
    if any(w in text for w in ['food', 'snack', 'cookie', 'chip', 'candy', 'chocolate', 'cereal']):
        category = 'food'
    elif any(w in text for w in ['shampoo', 'soap', 'lotion', 'cream', 'beauty', 'cosmetic']):
        category = 'beauty'
    elif any(w in text for w in ['vitamin', 'supplement', 'pill', 'capsule', 'tablet']):
        category = 'supplement'
    elif any(w in text for w in ['beverage', 'drink', 'soda', 'juice', 'water', 'coffee', 'tea']):
        category = 'beverage'
    elif any(w in text for w in ['detergent', 'cleaner', 'cleaning', 'laundry']):
        category = 'household'

    weight_map = {
        'oz': 28.35, 'ounce': 28.35, 'ounces': 28.35,
        'lb': 453.59, 'lbs': 453.59, 'pound': 453.59, 'pounds': 453.59,
        'kg': 1000, 'kilogram': 1000, 'kilograms': 1000,
        'g': 1, 'gram': 1, 'grams': 1,
        'mg': 0.001, 'milligram': 0.001
    }

    volume_map = {
        'l': 1000, 'liter': 1000, 'litre': 1000, 'liters': 1000,
        'ml': 1, 'milliliter': 1, 'milliliters': 1,
        'fl oz': 29.57, 'floz': 29.57, 'fluid ounce': 29.57, 'fl. oz': 29.57,
        'gal': 3785, 'gallon': 3785, 'gallons': 3785,
        'qt': 946, 'quart': 946, 'quarts': 946,
        'cup': 237, 'cups': 237, 'pint': 473, 'pints': 473
    }

    count_units = ['count', 'ct', 'pack', 'packs', 'piece', 'pieces', 'pc', 'each', 'ea', 'item', 'items']

    for k, v in weight_map.items():
        if k in unit or unit == k:
            return round(value * v, 3), 'g', category

    for k, v in volume_map.items():
        if k in unit or unit == k:
            return round(value * v, 3), 'ml', category

    for k in count_units:
        if k in unit or unit == k:
            return value, 'count', category

    return value, unit, category

def extract_numerical_features(text):
    if pd.isna(text):
        return []
    numbers = re.findall(r'\d+\.?\d*', str(text))
    return [float(n) for n in numbers if float(n) < 10000]

def extract_premium_budget_signals(text):
    if pd.isna(text):
        return 0

    text = str(text).lower()

    premium = {
        'organic': 10, 'premium': 8, 'gourmet': 10, 'luxury': 12, 'deluxe': 8,
        'professional': 6, 'ultra': 6, 'supreme': 8, 'artisan': 9, 'imported': 7,
        'natural': 5, 'authentic': 5, 'handmade': 8, 'specialty': 6, 'pure': 5,
        'extra virgin': 9, 'cold pressed': 8, 'grass fed': 9, 'free range': 8,
        'certified': 6, 'award': 7, 'finest': 8, 'select': 5, 'choice': 5,
        'prime': 9, 'aged': 7, 'reserve': 8, 'signature': 6, 'exclusive': 8,
        'boutique': 9, 'craft': 7, 'small batch': 8, 'heirloom': 7, 'raw': 6,
        'whole': 4, 'fresh': 4, 'estate': 7, 'single origin': 8, 'fair trade': 6
    }

    budget = {
        'value': -5, 'economy': -7, 'budget': -7, 'basic': -5, 'standard': -4,
        'generic': -6, 'store brand': -7, 'great value': -6, 'everyday': -4,
        'simple': -3, 'classic': -2, 'regular': -3, 'plain': -4, 'bulk': -5
    }

    score = 0
    for word, weight in premium.items():
        score += weight * len(re.findall(r'\b' + re.escape(word) + r'\b', text))
    for word, weight in budget.items():
        score += weight * len(re.findall(r'\b' + re.escape(word) + r'\b', text))

    return score

def extract_advanced_features(df, is_train=True, artifacts=None):
    print(f"\nProcessing {len(df)} samples...")
    df = df.copy()

    print("  Extracting fields...")
    extracted = df['catalog_content'].apply(extract_enhanced_fields)

    df['item_name'] = extracted.apply(lambda x: x[0].get('item', ''))
    df['bullets'] = extracted.apply(lambda x: x[0].get('bullets', ''))
    df['description'] = extracted.apply(lambda x: x[0].get('description', ''))
    df['brand'] = extracted.apply(lambda x: x[0].get('brand', ''))
    df['raw_value'] = extracted.apply(lambda x: x[1])
    df['raw_unit'] = extracted.apply(lambda x: x[2])

    df['full_text'] = (df['item_name'].fillna('') + ' ' +
                       df['description'].fillna('') + ' ' +
                       df['bullets'].fillna('') + ' ' +
                       df['brand'].fillna(''))

    print("  Standardizing units...")
    std_results = df.apply(
        lambda x: advanced_unit_standardization(x['raw_value'], x['raw_unit'], x['full_text']),
        axis=1
    )

    df['value'] = std_results.apply(lambda x: x[0])
    df['unit'] = std_results.apply(lambda x: x[1])
    df['category'] = std_results.apply(lambda x: x[2])

    if is_train:
        artifacts = artifacts or {}
        cat_unit_medians = df.groupby(['category', 'unit'])['value'].median().to_dict()
        cat_medians = df.groupby('category')['value'].median().to_dict()
        global_median = df['value'].median()

        artifacts['cat_unit_medians'] = cat_unit_medians
        artifacts['cat_medians'] = cat_medians
        artifacts['global_median'] = global_median

    def impute_value(row):
        if pd.notna(row['value']):
            return row['value']
        key = (row['category'], row['unit'])
        if key in artifacts['cat_unit_medians']:
            return artifacts['cat_unit_medians'][key]
        if row['category'] in artifacts['cat_medians']:
            return artifacts['cat_medians'][row['category']]
        return artifacts['global_median']

    df['value'] = df.apply(impute_value, axis=1)

    df['value_log'] = np.log1p(df['value'])
    df['value_log2'] = np.log1p(df['value_log'])
    df['value_sqrt'] = np.sqrt(df['value'])
    df['value_cbrt'] = np.cbrt(df['value'])
    df['value_squared'] = df['value'] ** 2
    df['value_inv'] = 1 / (df['value'] + 1)

    df['name_len'] = df['item_name'].fillna('').str.len()
    df['desc_len'] = df['description'].fillna('').str.len()
    df['bullet_len'] = df['bullets'].fillna('').str.len()
    df['brand_len'] = df['brand'].fillna('').str.len()
    df['total_text_len'] = df['name_len'] + df['desc_len'] + df['bullet_len']

    df['word_count'] = df['full_text'].str.split().str.len()
    df['unique_words'] = df['full_text'].apply(lambda x: len(set(str(x).lower().split())))
    df['word_diversity'] = df['unique_words'] / (df['word_count'] + 1)
    df['avg_word_len'] = df['full_text'].apply(lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0)

    df['premium_score'] = df['full_text'].apply(extract_premium_budget_signals)

    df['pack_size'] = df['full_text'].apply(
        lambda x: int(m.group(1)) if (m := re.search(r'(\d+)\s*(?:pack|count|ct\b)', str(x).lower())) else 1
    )
    df['pack_size'] = df['pack_size'].clip(1, 200)
    df['is_multipack'] = (df['pack_size'] > 1).astype(int)

    df['numbers_in_text'] = df['full_text'].apply(extract_numerical_features)
    df['num_count'] = df['numbers_in_text'].apply(len)
    df['max_number'] = df['numbers_in_text'].apply(lambda x: max(x) if x else 0)
    df['min_number'] = df['numbers_in_text'].apply(lambda x: min(x) if x else 0)
    df['sum_numbers'] = df['numbers_in_text'].apply(lambda x: sum(x) if x else 0)

    df['value_per_pack'] = df['value'] / df['pack_size']
    df['log_value_per_pack'] = np.log1p(df['value_per_pack'])
    df['text_density'] = df['total_text_len'] / (df['value'] + 1)
    df['premium_per_value'] = df['premium_score'] / (df['value'] + 1)

    if is_train:
        brand_freq = df['brand'].value_counts().to_dict()
        artifacts['brand_freq'] = brand_freq

    df['brand_freq'] = df['brand'].map(artifacts['brand_freq']).fillna(0)
    df['is_rare_brand'] = (df['brand_freq'] < 5).astype(int)

    cat_dummies = pd.get_dummies(df['category'], prefix='cat', dtype=np.float32)
    unit_dummies = pd.get_dummies(df['unit'], prefix='unit', dtype=np.float32)

    if is_train:
        artifacts['cat_cols'] = cat_dummies.columns.tolist()
        artifacts['unit_cols'] = unit_dummies.columns.tolist()
    else:
        for col in artifacts['cat_cols']:
            if col not in cat_dummies.columns:
                cat_dummies[col] = 0
        for col in artifacts['unit_cols']:
            if col not in unit_dummies.columns:
                unit_dummies[col] = 0
        cat_dummies = cat_dummies[artifacts['cat_cols']]
        unit_dummies = unit_dummies[artifacts['unit_cols']]

    df = pd.concat([df, cat_dummies, unit_dummies], axis=1)

    print("  Creating interactions...")
    df['value_x_premium'] = df['value'] * df['premium_score']
    df['log_value_x_premium'] = df['value_log'] * df['premium_score']
    df['value_x_pack'] = df['value'] * df['pack_size']
    df['sqrt_value_x_pack'] = df['value_sqrt'] * df['pack_size']
    df['value_x_word_div'] = df['value'] * df['word_diversity']
    df['value_x_brand_freq'] = df['value'] * np.log1p(df['brand_freq'])

    for col in cat_dummies.columns:
        df[f'{col}_x_value'] = df[col] * df['value']

    if is_train:
        qt = QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(df)))
        df['value_quantile'] = qt.fit_transform(df[['value']])
        artifacts['quantile_transformer'] = qt
    else:
        df['value_quantile'] = artifacts['quantile_transformer'].transform(df[['value']])

    df['quantile_x_premium'] = df['value_quantile'] * df['premium_score']

    return df, artifacts

def create_embeddings_enhanced(df, is_train=True, artifacts=None):
    print("\nCreating embeddings...")

    if is_train:
        model = SentenceTransformer(CONFIG['embedding_model'])
        artifacts = artifacts or {}
        artifacts['embedding_model'] = model
    else:
        model = artifacts['embedding_model']

    combined = (df['item_name'].fillna('') + ' ' +
                df['bullets'].str[:400].fillna('') + ' ' +
                df['brand'].fillna(''))

    embeddings = model.encode(combined.tolist(), batch_size=128,
                             show_progress_bar=False, normalize_embeddings=True)

    if is_train:
        svd = TruncatedSVD(n_components=CONFIG['embedding_dims'], random_state=42)
        emb_reduced = svd.fit_transform(embeddings)
        artifacts['svd'] = svd
    else:
        emb_reduced = artifacts['svd'].transform(embeddings)

    for i in range(emb_reduced.shape[1]):
        df[f'emb_{i}'] = emb_reduced[:, i]

    if is_train:
        tfidf = TfidfVectorizer(max_features=CONFIG['tfidf_features'],
                               ngram_range=(1, 2), min_df=2, max_df=0.95,
                               sublinear_tf=True)
        tfidf_mat = tfidf.fit_transform(df['full_text'].fillna(''))
        artifacts['tfidf'] = tfidf
    else:
        tfidf_mat = artifacts['tfidf'].transform(df['full_text'].fillna(''))

    tfidf_dense = tfidf_mat.toarray()
    for i in range(tfidf_dense.shape[1]):
        df[f'tfidf_{i}'] = tfidf_dense[:, i]

    return df, artifacts

def main():
    print("\nADVANCED PRICE PREDICTION MODEL")

    print("\nLoading data...")
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')

    print(f"Train: {train.shape}, Test: {test.shape}")
    print(f"Price stats:\n{train['price'].describe()}")

    y = train['price'].values
    y_log = np.log1p(y)

    print("\nFEATURE ENGINEERING")

    train_proc, artifacts = extract_advanced_features(train, is_train=True)
    train_proc, artifacts = create_embeddings_enhanced(train_proc, is_train=True, artifacts=artifacts)

    test_proc, _ = extract_advanced_features(test, is_train=False, artifacts=artifacts)
    test_proc, _ = create_embeddings_enhanced(test_proc, is_train=False, artifacts=artifacts)

    drop_cols = ['catalog_content', 'item_name', 'description', 'bullets',
                'brand', 'full_text', 'raw_value', 'raw_unit', 'unit',
                'category', 'image_link', 'numbers_in_text']

    train_proc = train_proc.drop(columns=[c for c in drop_cols if c in train_proc.columns], errors='ignore')
    test_proc = test_proc.drop(columns=[c for c in drop_cols if c in test_proc.columns], errors='ignore')

    feat_cols = [c for c in train_proc.columns if c not in ['sample_id', 'price']]
    for col in feat_cols:
        if col not in test_proc.columns:
            test_proc[col] = 0

    X_train = train_proc[feat_cols].fillna(0).replace([np.inf, -np.inf], 0).values
    X_test = test_proc[feat_cols].fillna(0).replace([np.inf, -np.inf], 0).values

    print(f"\nFinal feature shape: {X_train.shape}")

    print("\nMODEL TRAINING")

    bins = pd.qcut(y, q=10, labels=False, duplicates='drop')
    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=42)

    xgb_oof = np.zeros(len(X_train))
    lgb_oof = np.zeros(len(X_train))
    cat_oof = np.zeros(len(X_train))

    xgb_test_preds = []
    lgb_test_preds = []
    cat_test_preds = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, bins)):
        print(f"\nFold {fold+1}/{CONFIG['n_folds']}")

        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_log[tr_idx], y_log[val_idx]

        xgb_params = {
            'objective': 'reg:squarederror',
            'learning_rate': 0.02,
            'max_depth': 7,
            'min_child_weight': 3,
            'subsample': 0.8,
            'colsample_bytree': 0.7,
            'gamma': 3,
            'reg_alpha': 3,
            'reg_lambda': 8,
            'random_state': 42,
            'tree_method': 'hist',
        }

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        dtest = xgb.DMatrix(X_test)

        xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=CONFIG['n_estimators'],
                             evals=[(dval, 'val')], early_stopping_rounds=CONFIG['early_stopping'],
                             verbose_eval=False)

        xgb_oof[val_idx] = xgb_model.predict(dval)
        xgb_test_preds.append(xgb_model.predict(dtest))

        lgb_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'learning_rate': 0.02,
            'num_leaves': 63,
            'min_data_in_leaf': 30,
            'feature_fraction': 0.7,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'lambda_l1': 3,
            'lambda_l2': 8,
            'random_state': 42,
            'verbose': -1
        }

        lgb_train = lgb.Dataset(X_tr, label=y_tr)
        lgb_val = lgb.Dataset(X_val, label=y_val)

        lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=CONFIG['n_estimators'],
                             valid_sets=[lgb_val],
                             callbacks=[lgb.early_stopping(CONFIG['early_stopping']), lgb.log_evaluation(0)])

        lgb_oof[val_idx] = lgb_model.predict(X_val)
        lgb_test_preds.append(lgb_model.predict(X_test))

        cat_model = cb.CatBoostRegressor(
            loss_function='RMSE',
            learning_rate=0.02,
            depth=7,
            l2_leaf_reg=8,
            iterations=CONFIG['n_estimators'],
            early_stopping_rounds=CONFIG['early_stopping'],
            random_state=42,
            verbose=False
        )

        cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

        cat_oof[val_idx] = cat_model.predict(X_val)
        cat_test_preds.append(cat_model.predict(X_test))

        xgb_smape = calculate_smape(np.expm1(y_val), np.expm1(xgb_oof[val_idx]))
        lgb_smape = calculate_smape(np.expm1(y_val), np.expm1(lgb_oof[val_idx]))
        cat_smape = calculate_smape(np.expm1(y_val), np.expm1(cat_oof[val_idx]))

        ensemble = (xgb_oof[val_idx] + lgb_oof[val_idx] + cat_oof[val_idx]) / 3
        fold_smape = calculate_smape(np.expm1(y_val), np.expm1(ensemble))

        print(f"  XGB: {xgb_smape:.4f} | LGB: {lgb_smape:.4f} | CAT: {cat_smape:.4f}")
        print(f"  Ensemble: {fold_smape:.4f}")

    print("\nFINAL RESULTS")

    final_oof = (xgb_oof + lgb_oof + cat_oof) / 3
    final_smape = calculate_smape(y, np.expm1(final_oof))

    print(f"\nOOF SMAPE: {final_smape:.4f}")

    xgb_smape = calculate_smape(y, np.expm1(xgb_oof))
    lgb_smape = calculate_smape(y, np.expm1(lgb_oof))
    cat_smape = calculate_smape(y, np.expm1(cat_oof))

    print(f"Individual - XGB: {xgb_smape:.4f} | LGB: {lgb_smape:.4f} | CAT: {cat_smape:.4f}")

    xgb_test = np.mean(xgb_test_preds, axis=0)
    lgb_test = np.mean(lgb_test_preds, axis=0)
    cat_test = np.mean(cat_test_preds, axis=0)

    final_test = (xgb_test + lgb_test + cat_test) / 3
    predictions = np.expm1(final_test)

    submission = pd.DataFrame({
        'sample_id': test['sample_id'],
        'price': predictions
    })

    submission.to_csv('submission.csv', index=False)

    print("\nSUBMISSION CREATED")
    print(f"Price range: ${predictions.min():.2f} - ${predictions.max():.2f}")
    print(f"Mean price: ${predictions.mean():.2f}")
    print(f"Median price: ${np.median(predictions):.2f}")
    print("\nFirst 10 predictions:")
    print(submission.head(10))

if __name__ == '__main__':
    main()